# 📓 Semana 12 · Dia 5 — LangChain → LangGraph: primeiro grafo com estado

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | GenAI Engineer Associate |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Primeiro fluxo LangGraph funcionando |

---


## 📖 Teoria — Chain vs Graph

Uma **chain** é uma sequência fixa (retrieve → generate). Um **grafo (LangGraph)** tem **nós** (funções), **arestas** (transições) e **estado** (memória entre nós) — e pode **decidir** o próximo passo:

```
início → nó_1 (rota) → nó_A ou nó_B → fim
```


## 📖 Teoria — Componentes do LangGraph

- **State**: dict tipado que flui entre nós
- **Node**: função que recebe estado e retorna updates
- **Edge**: ligação entre nós; `conditional_edge` decide o caminho
- **Graph**: compila e executa (`graph.invoke`)


### 💻 Na prática — Primeiro grafo

Crie um grafo com rota condicional (pergunta de vendas vs geral).


In [ ]:
# Dependências
from langgraph.graph import StateGraph, END
from typing import TypedDict

class Estado(TypedDict):
    pergunta: str
    resposta: str

In [ ]:
# Nós
def rotear(estado):
    p = estado["pergunta"].lower()
    if any(k in p for k in ["venda", "receita", "produto", "país"]):
        return {"destino": "vendas"}
    return {"destino": "geral"}

def responder_vendas(estado):
    return {"resposta": "[RAG de vendas] " + estado["pergunta"]}

def responder_geral(estado):
    return {"resposta": "[Assistente geral] " + estado["pergunta"]}

print("Nós definidos: rotear, responder_vendas, responder_geral")

In [ ]:
# Montar o grafo com aresta condicional
g = StateGraph(Estado)
g.add_node("rotear", rotear)
g.add_node("vendas", responder_vendas)
g.add_node("geral", responder_geral)
g.set_entry_point("rotear")
g.add_conditional_edges("rotear",
    lambda e: "vendas" if e.get("destino") == "vendas" else "geral")
g.add_edge("vendas", END)
g.add_edge("geral", END)
app = g.compile()
print("Grafo compilado.")

In [ ]:
# Executar
r1 = app.invoke({"pergunta": "Qual a receita de novembro?"})
r2 = app.invoke({"pergunta": "Oi, tudo bem?"})
print("1:", r1["resposta"])
print("2:", r2["resposta"])

### 💻 Na prática — Conectando o RAG como nó

Troque `responder_vendas` pela chain RAG da Semana 11 — o agente de dados começa a nascer.


In [ ]:
# Nó de vendas usando o RAG
def responder_vendas_rag(estado):
    r = rag.invoke({"input": estado["pergunta"]})
    return {"resposta": r["answer"]}
print("Substitua responder_vendas por esta função no grafo.")

> 🎯 **Dica de prova**: GenAI Assoc/agentes: nodes + edges + state + conditional_edge são o vocabulário do LangGraph. Pergunta: 'como fazer o agente escolher a ferramenta?' → conditional edge / tool calling.


## 🎯 Exercícios de fixação

**1.** Adicione um terceiro nó de rota (ex.: 'estoque').

**2.** Qual a diferença entre edge normal e conditional_edge?

**3.** O que o estado carrega entre os nós?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Terceiro nó

Adicione `estoque` ao rotear e uma aresta condicional para ele.

**2.** Edges

Normal: sempre segue. Conditional: função decide qual nó vem — base do raciocínio de agentes.

**3.** Estado

Tudo que os nós precisam compartilhar: pergunta, contexto, histórico, resposta parcial — definido no TypedDict.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*